In [1]:
pip install pandas plotly nbformat nbclient ipywidgets matplotlib "urllib3<2"

You should consider upgrading via the '/Users/stripura/Desktop/ocp-workload-analysis-/venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. THE NAMESPACE GRADING FUNCTION
def assign_ns_tier(row):
    score = 0
    
    # Workload Count (Deployments + StatefulSets)
    workloads = row.get('Deployments', 0) + row.get('StatefulSets', 0)
    score += (workloads * 1)
    
    # Pod Density
    pods = row.get('Pods', 0)
    score += (pods * 0.5)
    
    # Storage Complexity (PVCs)
    pvcs = row.get('PVCs', 0)
    score += (pvcs * 3)
    
    # Networking & Security Overhead
    net_ops = row.get('NetworkPolicies', 0) + row.get('Ingress', 0)
    score += (net_ops * 2)

    # Final Namespace Grading
    # Adjust these thresholds based on your cluster size
    if score >= 50:
        return 'Large'
    elif score >= 15:
        return 'Medium'
    else:
        return 'Small'

# 2. THE NAMESPACE DASHBOARD HELPER
def create_ns_complexity_dashboard(df):
    """
    Standardized helper to generate Pie and Bar charts for Namespaces.
    """
    # Apply the Tiering Logic
    df['NS_Complexity_Tier'] = df.apply(assign_ns_tier, axis=1)
    
    # Prepare Data
    tier_order = ['Small', 'Medium', 'Large']
    counts = df['NS_Complexity_Tier'].value_counts().reindex(tier_order, fill_value=0)
    colors = ['#8BC34A', '#2196F3', '#673AB7'] # Different color palette (Green, Blue, Purple)
    
    # Setup Figure
    fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(16, 7))
    
    # --- PIE CHART ---
    counts.plot(kind='pie', ax=ax_pie, autopct='%1.1f%%', colors=colors, startangle=140,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2}, textprops={'fontweight': 'bold'})
    ax_pie.set_title('Namespace Complexity Share', fontsize=14, pad=15)
    ax_pie.set_ylabel('')

    # --- BAR CHART ---
    counts.plot(kind='bar', ax=ax_bar, color=colors, edgecolor='black', alpha=0.8)
    ax_bar.set_title('Namespace Counts by Tier', fontsize=14, pad=15)
    ax_bar.set_xlabel('Namespace Complexity', fontsize=12)
    ax_bar.set_ylabel('Number of Namespaces', fontsize=12)
    ax_bar.grid(axis='y', linestyle='--', alpha=0.4)
    
    # Add labels
    for i, v in enumerate(counts):
        ax_bar.text(i, v + (max(counts) * 0.02), str(v), ha='center', fontweight='bold')

    plt.suptitle('Kubernetes Namespace Complexity Analysis', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [3]:
# --- HOW TO RUN ---
df_ns = pd.read_csv('sample_data/k8s_ns_100_sample.csv')
create_ns_complexity_dashboard(df_ns)

FileNotFoundError: [Errno 2] No such file or directory: 'sample_data/k8s_ns_100_sample.csv'